In [9]:
'''Load a CSV file containing daily temperature data for Ahmedabad and create two new features: a 3-day lag of the temperature and a 7-day rolling average of the temperature using pandas.'''

import pandas as pd

df = pd.read_csv("ahmedabad_daily_temperature.csv")

# Convert Date
df["Date"] = pd.to_datetime(
    df["Date"],
    format="mixed",
    dayfirst=True
)

# Convert Temp Max to numeric
# Invalid values such as '-----' become NaN
df["Temp Max"] = pd.to_numeric(df["Temp Max"], errors="coerce")

# Sort by date
df = df.sort_values("Date")

# 3-day lag
df["Temp_Max_Lag_3"] = df["Temp Max"].shift(3)

# 7-day rolling average
df["Temp_Max_Rolling_7"] = df["Temp Max"].rolling(window=7).mean()

print(df.head(10))

        Date Rain   Temp Max     Temp Min           Location  Rain.1  \
0 1951-01-01    0  25.059999  6.590000153  POINT (72.5 23.5)     0.0   
1 1951-01-02    0  25.910000  7.510000229  POINT (72.5 23.5)     0.0   
2 1951-01-03    0  26.469999  8.529999733  POINT (72.5 23.5)     0.0   
3 1951-01-04    0  26.809999  8.840000153  POINT (72.5 23.5)     0.0   
4 1951-01-05    0  26.280001  8.479999542  POINT (72.5 23.5)     0.0   
5 1951-01-06    0  26.520000  11.02999973  POINT (72.5 23.5)     0.0   
6 1951-01-07    0  27.700001  10.81000042  POINT (72.5 23.5)     0.0   
7 1951-01-08    0  27.320000  10.53999996  POINT (72.5 23.5)     0.0   
8 1951-01-09    0  26.330000  8.680000305  POINT (72.5 23.5)     0.0   
9 1951-01-10    0  26.580000  9.010000229  POINT (72.5 23.5)     0.0   

   Temp Max.1  Temp Min.1         Location.1  Temp_Max_Lag_3  \
0   27.860001       10.54  POINT (72.5 22.5)             NaN   
1   28.760000       10.57  POINT (72.5 22.5)             NaN   
2   29.969999  

In [11]:
import pandas as pd

data = {
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
    "product": [
        "Laptop", "Mobile", "Headphones", "Keyboard", "Mouse",
        "Monitor", "Tablet", "Smartwatch", "Speaker", "Power Bank"
    ],
    "price": [55000, 25000, 1500, 1200, 800, 15000, 22000, 5000, 3000, 1200],
    "order_date": [
        "2026-01-05",
        "2026-01-10",
        "2026-01-11",
        "2026-01-16",
        "2026-01-18",
        "2026-01-24",
        "2026-02-02",
        "2026-02-08",
        "2026-02-14",
        "2026-02-22"
    ]
}

df = pd.DataFrame(data)

# Save as CSV
df.to_csv("flipkart_sales.csv", index=False)

print(df)

   order_id     product  price  order_date
0      1001      Laptop  55000  2026-01-05
1      1002      Mobile  25000  2026-01-10
2      1003  Headphones   1500  2026-01-11
3      1004    Keyboard   1200  2026-01-16
4      1005       Mouse    800  2026-01-18
5      1006     Monitor  15000  2026-01-24
6      1007      Tablet  22000  2026-02-02
7      1008  Smartwatch   5000  2026-02-08
8      1009     Speaker   3000  2026-02-14
9      1010  Power Bank   1200  2026-02-22


In [12]:
'''Extract date-time features from a Flipkart-style sales dataset (order_date column): create new columns for day of week, month, and whether the day is a weekend using pandas.'''

import pandas as pd

# Load dataset
df = pd.read_csv("flipkart_sales.csv")

# Convert order_date to datetime
df["order_date"] = pd.to_datetime(df["order_date"])

# Create day of week
df["day_of_week"] = df["order_date"].dt.day_name()

# Create month
df["month"] = df["order_date"].dt.month

# Check if weekend
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5

print(df.head())

   order_id     product  price order_date day_of_week  month  is_weekend
0      1001      Laptop  55000 2026-01-05      Monday      1       False
1      1002      Mobile  25000 2026-01-10    Saturday      1        True
2      1003  Headphones   1500 2026-01-11      Sunday      1        True
3      1004    Keyboard   1200 2026-01-16      Friday      1       False
4      1005       Mouse    800 2026-01-18      Sunday      1        True


In [14]:
'''Train a Random Forest Regressor to forecast the next day's temperature using the lag and rolling features you created earlier. Print the predicted value for the next available date.'''

import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# Load data
df = pd.read_csv("ahmedabad_daily_temperature.csv")

# Convert date
df["Date"] = pd.to_datetime(
    df["Date"],
    format="mixed",
    dayfirst=True
)

# Convert temperature to numeric
df["Temp Max"] = pd.to_numeric(
    df["Temp Max"],
    errors="coerce"
)

# Sort by date
df = df.sort_values("Date")

# Create 3-day lag
df["Temp_Max_Lag_3"] = df["Temp Max"].shift(3)

# Create 7-day rolling average
df["Temp_Max_Rolling_7"] = (
    df["Temp Max"].rolling(window=7).mean()
)

# Remove rows with missing values
df = df.dropna()

# Features
X = df[["Temp_Max_Lag_3", "Temp_Max_Rolling_7"]]

# Target
y = df["Temp Max"]

# Train Random Forest
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X, y)

# Use the latest available features to predict next day's temperature
latest_features = X.iloc[[-1]]

predicted_temp = model.predict(latest_features)[0]

# Next available date
next_date = df["Date"].iloc[-1] + pd.Timedelta(days=1)

print("Next Date:", next_date.date())
print("Predicted Temperature:", round(predicted_temp, 2))

Next Date: 1953-04-20
Predicted Temperature: 34.6


In [17]:
'''Calculate MAE, RMSE, and MAPE between your model's predictions and the actual values for the test set. Print all three metrics and explain in one line which metric is most sensitive to large errors.<br><br><em><strong>Hint:</strong> Use sklearn.metrics.mean_absolute_error, mean_squared_error, and write a simple function for MAPE.</em>'''

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load data
df = pd.read_csv("ahmedabad_daily_temperature.csv")

# Convert date
df["Date"] = pd.to_datetime(
    df["Date"],
    format="mixed",
    dayfirst=True
)

# Convert temperature to numeric
df["Temp Max"] = pd.to_numeric(
    df["Temp Max"],
    errors="coerce"
)

# Sort by date
df = df.sort_values("Date")

# Create features
df["Temp_Max_Lag_3"] = df["Temp Max"].shift(3)

df["Temp_Max_Rolling_7"] = (
    df["Temp Max"].rolling(7).mean()
)

# Remove missing values
df = df.dropna()

# Features and target
X = df[["Temp_Max_Lag_3", "Temp_Max_Rolling_7"]]
y = df["Temp Max"]

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Train model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

# Predict test data
y_pred = model.predict(X_test)

# MAE
mae = mean_absolute_error(y_test, y_pred)

# RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# MAPE
mape = np.mean(
    np.abs((y_test - y_pred) / y_test)
) * 100

# Print results
print("MAE:", mae)
print("RMSE:", rmse)
print("MAPE:", mape, "%")

print("RMSE is most sensitive to large errors because it squares the errors.")

MAE: 1.1484771633269453
RMSE: 1.552660898526525
MAPE: 3.658820926900462 %
RMSE is most sensitive to large errors because it squares the errors.
